# Data Profiling

A new batch of 5,000 transactions has arrived. Before it enters
Vaultline's systems, you need to profile it. What does the data actually
look like? Where are the problems hiding?

Profiling is the first thing you do with any new dataset. It is not
glamorous work, but it is the work that stops you from loading garbage
into production and then spending three days figuring out why the
dashboards are wrong.

The FCA does not accept "we didn't check" as an excuse.

## Setup

We need matplotlib for visualisations. Everything else comes with pandas.

In [ ]:
%pip install -q matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Loading the batch

The transaction batch is a CSV file that landed in our data directory
overnight. Let's load it and see what we're working with.

In [ ]:
df = pd.read_csv("../data/transactions_batch.csv")
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

## First pass: `.info()` and `.describe()`

These two methods are your opening move with any dataset. `.info()` tells
you the column names, data types, and how many non-null values each
column has. `.describe()` gives you summary statistics for the numeric
columns.

Together, they answer the question: "Is this data roughly what I expected?"

In [ ]:
df.info()

Already we can see something. Some columns have fewer non-null entries
than others. That means missing data. We'll quantify that properly in
a moment.

Now the numeric summary.

In [ ]:
df.describe()

Look at the `amount` column. The mean is somewhere around 45. The 75th
percentile is maybe 65 or so. But the max? It's enormous. That gap
between the 75th percentile and the max is a screaming red flag. We'll
come back to that.

## Categorical columns: `.value_counts()`

Numeric summaries are useful, but half the columns in this dataset are
categorical -- currency, status, merchant category, card type. For those,
`.value_counts()` is the right tool.

Let's see the distribution of currencies.

In [ ]:
df["currency"].value_counts()

Mostly GBP, some EUR and USD. That's what we'd expect for a UK fintech.
If we saw a large number of transactions in a currency we don't support,
that would be worth investigating.

Now transaction statuses.

In [ ]:
df["status"].value_counts()

In [ ]:
df["merchant_category"].value_counts()

Notice anything odd with merchant categories? Some rows have empty
strings rather than proper null values. That's a common data quality
issue -- the source system writes an empty string instead of omitting
the field. Pandas won't count those as NaN unless we tell it to.

## Null analysis

Missing data is one of the most common data quality problems, and one of
the most dangerous. A column that's 95% populated looks fine at a glance.
But that 5% might be systematically missing -- perhaps all the declined
transactions have no merchant category, or all the weekend transactions
have no customer ID.

Let's first replace empty strings with proper NaN values, then calculate
the null percentage for every column.

In [ ]:
# Replace empty strings with NaN so they show up in null analysis
df = df.replace("", pd.NA)

null_pct = df.isna().sum() / len(df) * 100
null_pct.sort_values(ascending=False)

Several columns have 5-8% missing values. Whether that's acceptable
depends on the column. A missing `merchant_category` is annoying but
probably not critical. A missing `customer_id` on a financial
transaction is a regulatory problem -- the FCA requires us to know
who made every transaction.

This is why profiling matters. The numbers themselves don't tell you
whether there's a problem. You need to combine the numbers with
domain knowledge.

## Distribution plots

Numbers are good. Pictures are better. Let's look at how transaction
amounts are distributed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df["amount"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("Transaction Amount Distribution")
axes[0].set_xlabel("Amount")
axes[0].set_ylabel("Frequency")

# Box plot
axes[1].boxplot(df["amount"].dropna(), vert=True)
axes[1].set_title("Transaction Amount Box Plot")
axes[1].set_ylabel("Amount")

plt.tight_layout()
plt.show()

## The histogram lie

That histogram looks roughly normal. Most transactions are small, which
is what we'd expect. The box plot shows some dots above the whiskers,
but the scale makes it hard to see what's going on.

Here's the problem: the histogram is hiding the outliers. When you have
values ranging from 1 to 500,000, a histogram with 50 bins puts nearly
everything into the first bin. The extreme values are invisible.

Let's look at the actual extreme values.

In [ ]:
df.nlargest(10, "amount")[["transaction_id", "amount", "currency", "merchant_name", "status"]]

There it is. Transactions of hundreds of thousands of pounds. The mean
is around 45. These values are orders of magnitude larger.

Are they errors? Fraud? Legitimate high-value transfers? We can't tell
from the data alone. But we absolutely need to flag them before they
enter the system. If one of those is a decimal point error (someone
typed 500000 instead of 50.00), it will corrupt every aggregate that
touches it -- daily totals, merchant averages, customer spending
summaries.

This is the lesson: **always check the extremes**. The histogram and
`.describe()` can look perfectly reasonable while hiding catastrophic
outliers. `nlargest()` and `nsmallest()` tell the real story.

Let's also see the histogram without the extreme outliers, so we can
actually understand the main distribution.

In [ ]:
# Filter to the main distribution (under 500)
normal_amounts = df[df["amount"] < 500]["amount"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(normal_amounts, bins=50, edgecolor="black", alpha=0.7)
ax.set_title("Transaction Amounts (under 500)")
ax.set_xlabel("Amount")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

Now we can see the actual shape of the distribution. Most transactions
cluster in the 10-80 range, with a long right tail.

## IQR-based outlier detection

Eyeballing charts is fine for exploration, but we need a systematic way
to flag outliers. The interquartile range (IQR) method is one of the
most common approaches.

The idea is straightforward:

1. Calculate Q1 (25th percentile) and Q3 (75th percentile)
2. IQR = Q3 - Q1
3. Lower bound = Q1 - 1.5 * IQR
4. Upper bound = Q3 + 1.5 * IQR
5. Anything outside those bounds is flagged as an outlier

The 1.5 multiplier is conventional. For financial data, you might use
a more aggressive multiplier (like 3) because legitimate transactions
can have wide variation.

In [ ]:
Q1 = df["amount"].quantile(0.25)
Q3 = df["amount"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")

In [ ]:
outliers = df[(df["amount"] < lower_bound) | (df["amount"] > upper_bound)]
print(f"Outliers found: {len(outliers)} ({len(outliers) / len(df) * 100:.1f}%)")
outliers[["transaction_id", "amount", "currency", "merchant_name"]].head(15)

The IQR method has flagged the extreme values, and probably some
moderately large but legitimate transactions too. That's the trade-off:
a tighter bound catches more problems but also flags more false
positives.

In practice, flagging doesn't mean rejecting. It means routing those
transactions to a human reviewer or a secondary validation step. The
goal is to make sure nothing slips through unexamined.

## Putting it together: a profiling summary

Let's write a function that produces a quick profiling report for any
DataFrame. This is the kind of utility function you'd keep in your
team's shared library.

In [ ]:
def profile_dataframe(df):
    """Print a quick profiling summary for a DataFrame."""
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"\nDuplicates: {df.duplicated().sum()}")
    print("\n--- Null Percentages ---")
    null_pct = df.isna().sum() / len(df) * 100
    for col, pct in null_pct.items():
        if pct > 0:
            print(f"  {col}: {pct:.1f}%")
    if null_pct.sum() == 0:
        print("  No nulls found")

    print("\n--- Numeric Columns ---")
    for col in df.select_dtypes(include="number").columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        n_outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
        print(f"  {col}: min={df[col].min()}, max={df[col].max()}, "
              f"mean={df[col].mean():.2f}, outliers={n_outliers}")

    print("\n--- Categorical Columns ---")
    for col in df.select_dtypes(include="object").columns:
        n_unique = df[col].nunique()
        print(f"  {col}: {n_unique} unique values")


profile_dataframe(df)

## Exercises

### Exercise 1

Use `.value_counts()` to examine the `card_type` column. Are there any
unexpected values? What percentage of rows have a missing card type?

### Exercise 2

Create a box plot showing the distribution of transaction amounts
grouped by `status` (completed, pending, declined, refunded). Do any
statuses have noticeably different distributions?

**Hint:** Filter to amounts under 500 first so the plot is readable.
You can use `df[df['amount'] < 500].boxplot(column='amount', by='status')`.

### Exercise 3

Write a function called `detect_outliers` that takes a DataFrame and a
column name, and returns a new DataFrame containing only the outlier
rows (using the IQR method with a 1.5 multiplier).

Test it on the `amount` column and verify it returns the same results
we found earlier.

### Exercise 4

Investigate whether the missing `customer_id` values are random or
systematic. Compare the `status` distribution for rows where
`customer_id` is null versus rows where it is present.

**Hint:** Use `df[df['customer_id'].isna()]['status'].value_counts()`
and compare with the overall distribution.

### Exercise 5

The 99th percentile of the `amount` column is a useful threshold for
flagging unusually large transactions. Calculate the 99th percentile
and filter the DataFrame to show all transactions above it. How many
are there? What merchants are involved?

## Summary

Profiling is not optional. It is the first line of defence against bad
data.

- **`.info()` and `.describe()`** are your opening move
- **`.value_counts()`** reveals the distribution of categorical columns
- **Null percentages** tell you where data is missing -- and you need
  domain knowledge to decide if that's a problem
- **Histograms lie** when outliers stretch the scale. Always use
  `nlargest()` and `nsmallest()` to check the extremes.
- **IQR outlier detection** gives you a systematic way to flag unusual
  values, but flagging is not the same as rejecting

In the next notebook, we'll move from detecting problems to preventing
them -- with proper schema validation using Pydantic.